In [ ]:
#Data cleaning extracted from RSF
import pandas as pd
from sklearn.model_selection import train_test_split
from sksurv.util import Surv

df = pd.read_csv('../Resources/Overall_cleaned.csv',encoding='latin-1')

In [ ]:
columns_to_keep = [
    'Regional nodes positive (1988+)',
    'Tumor Size Over Time Recode (1988+)',
    'Age recode with <1 year olds and 90+',
    'Total number of in situ/malignant tumors for patient',
    'SEER Combined Mets at DX-lung (2010+)',
    'Sex',
    'Year of diagnosis',
    'Regional nodes examined (1988+)',
    'SEER Combined Mets at DX-bone (2010+)',
    'Grade Recode (thru 2017)',
    'First malignant primary indicator',
    'Marital status at diagnosis',
    'Histology recode - broad groupings',
    'SEER Combined Mets at DX-brain (2010+)',
    'Primary Site - labeled',
    'SEER Combined Mets at DX-liver (2010+)',
    'AYA site recode 2020 Revision',
    'Tumor Size Summary (2016+)',
    'Rural-Urban Continuum Code',
    'Survival months',  # Keep target columns
    'Vital status recode (study cutoff used)'
]

df = df[columns_to_keep]
df = df.dropna()
print(df)

In [ ]:
y_time = df['Survival months']
y_event = df['Vital status recode (study cutoff used)']
X = df.drop(columns=['Survival months',
                     'Vital status recode (study cutoff used)'])

#create survival target (used for train test split only)
y_struct = Surv.from_arrays(event=y_event,time=y_time)

In [ ]:
#convert string columns to numeric (ie Female - 0)
from sklearn.preprocessing import LabelEncoder

#object implies text
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"categorical columns are {categorical_cols}")
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Only encode non-null values
    mask = X[col].notna()
    X.loc[mask,col] = le.fit_transform(X.loc[mask,col].astype(str))

    average_value = X.loc[mask,col].mode()[0]
    X.loc[~mask,col] = average_value
    #Convert to nullable integer to keep NaN
    X[col] = pd.to_numeric(X[col],errors = 'coerce').astype('Int64')

    #save encoder to see mapping
    label_encoders[col] = le

In [ ]:
#Split dataset into Train and Test
X_train,X_test,y_train,y_test = train_test_split(X,y_struct,test_size=0.2,random_state=42)

In [ ]:
#Libraries for DeepHit
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from pycox.models import DeepHitSingle
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
import torchtuples as tt

In [ ]:
#Extract durations and events from structured array
import numpy as np
durations_train = y_train['time'] 
events_train = y_train['event'].astype(np.float32) #convert from boolean to float since DeepHit reads numerical inputs
durations_test = y_test['time']
events_test = y_test['event'].astype(np.float32)    

In [ ]:
#Standardize non categorical features (so each feature has mean 0 and variance 1)
numerical_cols = X_train.columns.difference(categorical_cols)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

#Convert to float32 for PyTorch
X_train_scaled = X_train_scaled.values.astype('float32')
X_test_scaled = X_test_scaled.values.astype('float32')
#.values part converts data frame to numpy array, so it avoids errors when fitting model

In [ ]:
#Discretize time into bins (DeepHit requirement)
num_durations = 100 #number of bins

#create empty transformer object
labtrans = LabTransDiscreteTime(num_durations) #converts survival_months to bin index (which bin it falls into)

#return transformed y_train
y_train_discrete = labtrans.fit_transform(durations_train, events_train) 

#y_train_discrete is a list of indexes that each survival month belongs to, along with event status
index_of_train_duration, events = y_train_discrete


In [ ]:
#Create neural network
number_of_features = X_train_scaled.shape[1] 

#MLP stands for Multi-Layer Perceptron (standard feedforward neural network)
#creates neural network with fully connected layers
net = tt.practical.MLPVanilla(
    in_features=number_of_features,
    #hidden layer sizes
    num_nodes=[256, 128, 64], #output is 101 neurons - one per bin
    out_features=labtrans.out_features, #101 - number of output neurons
    batch_norm=True, #normalise activation between layers - output values of neurons
    dropout=0.3 #randomly turn of 30% neurons in each layer to reduce overfitting
)

In [ ]:
#Create DeepHit model

model = DeepHitSingle(
    net,
    #Optimiser that trains neural network by adjusting its weights
    #Uses momentum and RMSprop
    tt.optim.Adam(lr=0.001), #lr controls how big the weight updates are
    
    #hyper parameter that controls the value of likelihood loss (survival bin)
    alpha=0.2,
    #controls value of ranking loss (discrimination based on feature)
    sigma=0.1,
    duration_index=labtrans.cuts, #bin edges
)

In [ ]:
#check input data for extreme values
print("X_train_scaled stats:")
print(f"  Shape: {X_train_scaled.shape}")
print(f"  Min: {X_train_scaled.min()}")
print(f"  Max: {X_train_scaled.max()}")
print(f"  Mean: {X_train_scaled.mean()}")
print(f"  Std: {X_train_scaled.std()}")
print(f"  Contains NaN: {np.isnan(X_train_scaled).any()}")
print(f"  Contains Inf: {np.isinf(X_train_scaled).any()}")

#check specific rows
print("\nFirst 5 rows of X_train_scaled:")
print(X_train_scaled[:5])

#check for extreme outliers
print(f"\nPercentiles:")
print(f"  1st: {np.percentile(X_train_scaled, 1)}")
print(f"  99th: {np.percentile(X_train_scaled, 99)}")


In [ ]:
print(X_train_scaled)

In [ ]:
log = model.fit(
    X_train_scaled,
    y_train_discrete,
    #how many samples per weight update (ie train on chunks of 1024 patients)
    batch_size=1024,
    epochs=3,
    #show (print) training progress
    verbose=True
)


In [ ]:
print(log.to_pandas())

In [ ]:
import matplotlib.pyplot as plt
surv = model.predict_surv_df(X_test_scaled)
row = surv.iloc[40]
print(surv.index[40])
plt.figure(figsize=(8, 5))
plt.hist(row.values, bins=20, edgecolor='black')
plt.title('Histogram of Values at the 35 Month Time Point')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()


In [ ]:
#Test model and get c-index
#calculate C-index
from pycox.evaluation import EvalSurv

# Get survival predictions on test set
#test on the scaled version for consistency
surv = model.predict_surv_df(X_test_scaled)
print(f"surv: {surv}")

# Calculate C-index
ev = EvalSurv(surv, durations_test, events_test, censor_surv='km')
c_index = ev.concordance_td('antolini')

print(f"C-index: {c_index:.4f}")

#Time-dependent C-index
c_index_ipcw = ev.concordance_td('antolini')  
print(f"Time dependent C-index: {c_index_ipcw}")


In [ ]:
# # Bootstrap Aggregation (Bagging) for DeepHit
# from Bootstrap import BootstrapDeepHit

# bootstrap_model = BootstrapDeepHit(B=10, num_durations=100)
# bootstrap_model.fit(X_train_scaled, durations_train.values, events_train.values)

# # Evaluate bagged ensemble
# bootstrap_c_index = bootstrap_model.score(X_test_scaled, durations_test, events_test)
# print(f"Single model C-index: {c_index:.4f}")
# print(f"Bootstrap ensemble C-index: {bootstrap_c_index:.4f}")

# # Get averaged survival predictions for downstream use
# surv_bootstrap = bootstrap_model.predict_surv_df(X_test_scaled)

In [ ]:
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

# Define custom scorer for DeepHit (C-index)
def c_index_scorer(estimator, X, y):
    """
    Calculate C-index for survival model
    
    Parameters:
    - estimator: trained DeepHit model
    - X: feature matrix
    - y: structured array with 'time' and 'event' fields OR regular arrays
    """
    from pycox.evaluation import EvalSurv
    
    # Check if y is a structured array or regular array
    if hasattr(y, 'dtype') and y.dtype.names:
        # Structured array (from sksurv)
        durations = y['time']
        events = y['event']
    else:
        # If it's a tuple or other format
        try:
            durations, events = y
        except:
            raise ValueError(f"Unexpected y format: {type(y)}")
    
    # Ensure they're numpy arrays
    durations = np.asarray(durations)
    events = np.asarray(events)
    
    # Get survival predictions
    surv = estimator.predict_surv_df(X)
    
    # Calculate C-index
    ev = EvalSurv(surv, durations, events, censor_surv='km')
    c_index = ev.concordance_td('antolini')
    
    return c_index

# Calculate permutation importance
result = permutation_importance(
    model,                    # trained DeepHit model
    X_test_scaled[:1000],           # Test features (scaled)
    y_test[:1000],                  # Test target (structured array)
    n_repeats=10,            # Number of times to permute each feature
    random_state=42,
    scoring=c_index_scorer,  # Use C-index as metric
    n_jobs=-1                # Use all CPU cores
)

# Create DataFrame with results
perm_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values('importance_mean', ascending=False)

# Show top 20
print("\n" + "="*80)
print("top 20 most important features (permutation importance)")
print("="*80)
print(perm_importance.head(20).to_string(index=False))

# Show bottom 10 (likely useless features)
print("\n" + "="*80)
print("bottom 10 features")
print("="*80)
print(perm_importance.tail(10).to_string(index=False))

# Statistical significance check
# Features with importance > 2*std are statistically significant
perm_importance['significant'] = perm_importance['importance_mean'] > 2 * perm_importance['importance_std']

print(f"\nStatistically significant features: {perm_importance['significant'].sum()}")

# Visualize top 20
import matplotlib.pyplot as plt

top_20 = perm_importance.head(20)

plt.figure(figsize=(10, 8))
plt.barh(range(20), top_20['importance_mean'], xerr=top_20['importance_std'])
plt.yticks(range(20), top_20['feature'])
plt.xlabel('Permutation Importance (Change in C-index)')
plt.title('Top 20 Feature Importances - Permutation Method')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('deephit_permutation_importance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Get top 20 features
top_20 = perm_importance.head(20)

# Generate LaTeX table
latex_table = top_20.to_latex(
    index=False,
    caption='Top 20 Most Important Features (Permutation Importance)',
    label='tab:deephit_top20_importance',
    column_format='lrrr',  # left-align feature, right-align numbers, right-align for significant column
    float_format='%.6f',   # 6 decimal places
    escape=False           # Allow special characters
)

print(latex_table)

In [ ]:
"""
Analyse features

Create scatter plot for 'Age of Diagnosis' against 'Survival Months' Grouped by 'Event_Status'

"""

#allow x axis to change to unencoded values
def plotFeatureVsSurvival(feature_name, title_label, title_save, original_encoding=False):
    if original_encoding and feature_name in label_encoders:
        plot_data = pd.DataFrame({
            feature_name: label_encoders[feature_name].inverse_transform(df[feature_name]),
            'Survival_Months': y_train['time'],
            'Event_Status': y_train['event']
        })
    else:
        plot_data = pd.DataFrame({
            feature_name: X_train[feature_name],
            'Survival_Months': y_train['time'],
            'Event_Status': y_train['event']
        })
        
    #Create scatter plot
    plt.figure(figsize=(12,7))

    #plot censored patients
    censored = plot_data[plot_data['Event_Status'] == True]
    plt.scatter(censored[feature_name],censored['Survival_Months'],
                alpha=0.3,s=10,c='green',label='Dead', marker = 'o')

    plt.xlabel(title_label, fontsize=12)
    plt.ylabel('Survival Months',fontsize=12)
    plt.title(f'Survival Time vs {title_label}', fontsize=14, fontweight='bold')
    plt.legend(loc='best')
    # if original_encoding:
    #     plt.xticks(rotation=45, ha='right')  # 45-degree angle, right-aligned
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'Report_Resources/survival_vs_{title_save}_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def plotFeatureVsSurvival(feature_name, title_label, title_save, original_encoding=False):
    if original_encoding and feature_name in label_encoders:
        original_vals = label_encoders[feature_name].inverse_transform(
            X_train[feature_name].astype(int)
        )

        plot_data = pd.DataFrame({
            feature_name: original_vals,
            'Survival_Months': y_train['time'],
            'Event_Status': y_train['event']
        })
    else:
        plot_data = pd.DataFrame({
            feature_name: X_train[feature_name],
            'Survival_Months': y_train['time'],
            'Event_Status': y_train['event']
        })
    #Create scatter plot
    plt.figure(figsize=(12,7))

    #plot censored patients
    censored = plot_data[plot_data['Event_Status'] == True]
    plt.scatter(censored[feature_name],censored['Survival_Months'],
                alpha=0.3,s=10,c='green',label='Dead', marker = 'o')

    plt.xlabel(title_label, fontsize=12)
    plt.ylabel('Survival Months',fontsize=12)
    plt.title(f'Survival Time vs {title_label}', fontsize=14, fontweight='bold')
    plt.legend(loc='best')
    if original_encoding:
        plt.xticks(rotation=45, ha='right')  # 45-degree angle, right-aligned
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'Report_Resources/survival_vs_{title_save}_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plotFeatureVsSurvival("Regional Nodes Positive (1988+)","Survival against Regional Nodes Positive (1988+)","regional_nodes_positive_1988_survival",True)

In [ ]:
X_train.columns

In [ ]:
"""
Get bottom 10 important features
Check their covariance is constant (Done)

We have proved that permutation_importance function is accurate
"""

bot_10 = perm_importance.tail(10)
print(f"bot 10 features: {bot_10['feature']}")
bot_10_features = bot_10['feature'].tolist()
bot_10_df = df[bot_10_features]

#Remove constant features
constant_filter = X.nunique() == 1
X_constant= X.loc[:, constant_filter]
print(f"constant features are {X_constant.columns.tolist}")
constant_features = X_constant.columns.tolist()
for feature in constant_features:
    match = perm_importance[perm_importance['feature'] == feature]
    if not match.empty:
        rank = match.index[0] + 1 #Convert to 1-based rank
        importance = match['importance_mean'].values[0]
        print(f"importance of constant feature {feature} is {importance}")


In [ ]:
# Generate LaTeX table for Overleaf
latex_table = perm_importance.to_latex(
    index=False,
    caption='Top 20 Most Important Features (Permutation Importance)',
    label='tab:deephit_perm_importance',
    column_format='lrr',  # left-align feature names, right-align numbers
    float_format='%.6f',  # 6 decimal places
    escape=False
)

print(latex_table)


In [ ]:
#0 is female, 1 is male
male_df = df[df['Sex']=="Male"]
female_df = df[df['Sex']=="Female"]


male_survival = male_df[male_df['Vital status recode (study cutoff used)'] == 1]
male_survival_dist = male_survival['Survival months'].value_counts()

female_survival = female_df[female_df['Vital status recode (study cutoff used)'] == 0]
female_survival_dist = female_survival['Survival months'].value_counts()

plt.figure(figsize=(14, 7))

plt.hist(male_survival_dist, bins=50, alpha=0.5, label='Male (Deaths)', color='blue', edgecolor='black')
plt.hist(female_survival_dist, bins=50, alpha=0.5, label='Female (Deaths)', color='red', edgecolor='black')

plt.xlabel('Survival Months', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Survival Time Distribution by Sex (Deaths Only)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('Report_Resources/survival_histogram_by_sex.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Understanding feature data

#Use mapping to get value names from encoding
def encoding2Original(col):
    for encoded_value in X[col].unique():
        original_val = label_encoders[col].inverse_transform([encoded_value])[0]
        print(f"encoding {encoded_value} is {original_val}")

encoding2Original('SEER Combined Mets at DX-liver (2010+)')

In [ ]:
# Create risk labels based on patient surviving 119 months
def assign_risk_label(prob):
    """Assign EL1-EL10 based on survival probability"""
    if prob >= 0.9:
        return 'EL1'
    elif prob >= 0.8:
        return 'EL2'
    elif prob >= 0.7:
        return 'EL3'
    elif prob >= 0.6:
        return 'EL4'
    elif prob >= 0.5:
        return 'EL5'
    elif prob >= 0.4:
        return 'EL6'
    elif prob >= 0.3:
        return 'EL7'
    elif prob >= 0.2:
        return 'EL8'
    elif prob >= 0.1:
        return 'EL9'
    else:
        return 'EL10'

In [ ]:
#Produce KM plot of curve
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

def makeKMPlot(X_test_extended):
    #data used for KM plot
    km_data = pd.DataFrame({
        'time': X_test_extended['Survival_Time'],
        'event': X_test_extended['Event_Status'].astype(int), #0 - censored
        'risk_label': X_test_extended['Risk_Label']
    })

    #restrict plot to 10 years 
    max_time = 120
    km_data_10yr =km_data[km_data['time'] <= max_time]

    # Plot KM curves for each risk group
    plt.figure(figsize=(14,8))
    kmf = KaplanMeierFitter()

    #Define colours for each risk group
    color = ['#2ecc71', '#27ae60', '#f1c40f', '#f39c12', '#e67e22', 
            '#d35400', '#e74c3c', '#c0392b', '#8e44ad', '#2c3e50']

    risk_labels_sorted = [f'EL{i}' for i in range(1,11)]

    for i,label in enumerate(risk_labels_sorted):
        mask = km_data['risk_label'] == label
        n_patients = mask.sum()

        if n_patients > 0:
            kmf.fit(
                durations = km_data.loc[mask,'time'],
                event_observed = km_data.loc[mask,'event'],
                label = f'{label} (n={n_patients})'
                )
            kmf.plot_survival_function(
                color=color[i],
                linewidth=2.5,
                alpha=0.8
            )

In [ ]:
surv = model.predict_surv_df(X_test_scaled)
print(surv)
#calculate average survival probability across all time points for each patient
integrated_survival = surv.mean(axis=0).values

#high value - patient has high survival probability across most time points
#using integrated survival for creating labels is better as it considers more time points so avoids bias specifically at the 10 year time point

#still gives no representation for EL7 onwards
print(np.histogram(integrated_survival,bins=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]))

#ie EL1 represents bottom 10 percent of values, EL2 10 percent above
#this could mean that depending on data and not the algorithm there is still a lot of overlap between patients 

# risk_labels = pd.qcut(
#     1 - integrated_survival,  # Convert to risk score
#     q=10,
#     labels=[f'EL{i}' for i in range(1, 11)]
# )

#just X_test_scaled converted back into DataFrame with original column names restored
X_test_extended = pd.DataFrame(X_test_scaled,columns=X_test.columns)
print(f"X_test_extended shape: {X_test_extended.shape}")

risk_labels = [assign_risk_label(p) for p in integrated_survival]

X_test_extended['Risk_Label'] = risk_labels
X_test_extended['Survival_Time'] = durations_test
X_test_extended['Event_Status'] = events_test

makeKMPlot(X_test_extended)

In [ ]:
print(y_train_discrete)

In [ ]:
#Hyperparameter optimisation for KM plot 
import skopt.space.space as s

search_space = list()

search_space.append(s.Real(0.001,0.5,name='dropout'))
search_space.append(s.Real(0.001,0.1,name='lr'))
search_space.append(s.Real(0.001,1,name='alpha'))
search_space.append(s.Real(0.001,1,name='sigma'))
search_space.append(s.Categorical(['128,64','256,128,64','256,128,64,32','512,256,128'], name='num_nodes'))

#gibes precision score
from sklearn.model_selection import cross_val_score
#for cross validation
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from numpy import mean
from numpy import std
import numpy as np
import sys
import importlib
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

#To view in case something goes wrong
view_failed_X = pd.DataFrame()
view_failed_X_scaled = pd.DataFrame()
view_failed_y_discrete = pd.DataFrame()
view_failed_y = pd.DataFrame()
view_surv = pd.DataFrame()
view_surv_120 = pd.DataFrame()
def evaluate_model(params):
    global view_failed_X,view_failed_y,view_surv,view_failed_X_scaled,view_failed_y_discrete,view_surv_120
    dropout,lr,alpha,sigma,num_nodes = params
    #creates neural network with fully connected layers
    net = tt.practical.MLPVanilla(
        in_features=number_of_features,
        #hidden layer sizes
        num_nodes=[int(x) for x in num_nodes.split(',')], #output is 101 neurons - one per bin
        out_features=labtrans.out_features, #101 - number of output neurons
        batch_norm=True, #normalise activation between layers - output values of neurons
        dropout=dropout #randomly turn of 30% neurons in each layer to reduce overfitting
    )
    model = DeepHitSingle(
        net,
        #Optimiser that trains neural network by adjusting its weights
        #Uses momentum and RMSprop
        tt.optim.Adam(lr=lr), #lr controls how big the weight updates are
        
        #hyper parameter that controls the value of likelihood loss (survival bin)
        alpha=alpha,
        #controls value of ranking loss (discrimination based on feature)
        sigma=sigma,
        duration_index=labtrans.cuts #bin edges
    )
    
    #define the splits
    cv = RepeatedStratifiedKFold(n_splits=3,n_repeats=3,random_state=1)
    count=0
    scores = []
    #apply the splits 
    for train_idx, test_idx in cv.split(X_train,y_train):
        count+=1 #check that it's 30 iterations
        print(f"count is {count}")
        subset_X_train_scaled = X_train_scaled[train_idx]
        subset_y_train_discrete = (y_train_discrete[0][train_idx],y_train_discrete[1][train_idx])
        subset_X_test_scaled = X_train_scaled[test_idx]
        subset_y_test_discrete = (y_train_discrete[0][test_idx],y_train_discrete[1][test_idx])

        view_failed_X_scaled = subset_X_train_scaled
        view_failed_y_discrete = subset_y_train_discrete
        
        log = model.fit(
            subset_X_train_scaled,
            subset_y_train_discrete,
            #how many samples per weight update (ie train on chunks of 1024 patients)
            batch_size=1024,
            epochs=3,
            #show (print) training progress
            verbose=True
        )
        surv = model.predict_surv_df(subset_X_test_scaled)
        view_surv = surv
        # target_time = 120
        # closest_idx = np.abs(surv.index - target_time).argmin()
        # surv_at_120 = surv.iloc[closest_idx, :]
        # view_surv_120 = surv_at_120
        # risk_labels = surv_at_120.apply(assign_risk_label)

        integrated_survival = surv.mean(axis=0).values
        risk_labels = [assign_risk_label(p) for p in integrated_survival]

        subset_X_test_extended = subset_X_test_scaled.copy()
        subset_X_test_extended = pd.DataFrame(subset_X_test_scaled,columns=X_test.columns)
        subset_X_test_extended['Risk_Label'] = risk_labels
        print("counts of each risk label are: ")
        print(subset_X_test_extended['Risk_Label'].value_counts())
        view_failed_X = subset_X_test_extended
        view_failed_y = subset_y_test_discrete
        subset_X_test_extended['Survival_Time'] = subset_y_test_discrete[0]
        subset_X_test_extended['Event_Status'] = subset_y_test_discrete[1]

        km_plotter = KM_Plotter(subset_X_test_extended)
        print("checkpoint 1")
        km_plotter.makeKMPlot(plot=True)
        print("checkpoint 2")
        score = km_plotter.getKMScore()
        print("checkpoint 3")
        scores.append(score)
        print("checkpoint 4")
        print(f"Score for fold is: {score}")
    return(mean(scores))

import skopt
result = skopt.gp_minimize(evaluate_model,search_space,n_jobs=3, n_calls = 100)
print(f"Best accuracy: {1-result.fun}")
print(f"Best parameters: {result.x}")

In [ ]:
#Hyperparameter optimisation for KM plot 
import skopt.space.space as s

search_space = list()

search_space.append(s.Real(0.001,0.5,name='dropout'))
search_space.append(s.Real(0.001,0.1,name='lr'))
search_space.append(s.Real(0.001,1,name='alpha'))
search_space.append(s.Real(0.001,1,name='sigma'))
search_space.append(s.Categorical(['128,64','256,128,64','256,128,64,32','512,256,128'], name='num_nodes'))

#gibes precision score
from sklearn.model_selection import cross_val_score
#for cross validation
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from numpy import mean
from numpy import std
import numpy as np
import sys
import importlib
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

#To view in case something goes wrong
view_failed_X = pd.DataFrame()
view_failed_X_scaled = pd.DataFrame()
view_failed_y_discrete = pd.DataFrame()
view_failed_y = pd.DataFrame()
view_surv = pd.DataFrame()
view_surv_120 = pd.DataFrame()
scores = []
def evaluate_model(params, train_idx, test_idx):
    global view_failed_X,view_failed_y,view_surv,view_failed_X_scaled,view_failed_y_discrete,view_surv_120
    dropout,lr,alpha,sigma,num_nodes = params
    #creates neural network with fully connected layers
    net = tt.practical.MLPVanilla(
        in_features=number_of_features,
        #hidden layer sizes
        num_nodes=[int(x) for x in num_nodes.split(',')], #output is 101 neurons - one per bin
        out_features=labtrans.out_features, #101 - number of output neurons
        batch_norm=True, #normalise activation between layers - output values of neurons
        dropout=dropout #randomly turn of 30% neurons in each layer to reduce overfitting
    )
    model = DeepHitSingle(
        net,
        #Optimiser that trains neural network by adjusting its weights
        #Uses momentum and RMSprop
        tt.optim.Adam(lr=lr), #lr controls how big the weight updates are
        
        #hyper parameter that controls the value of likelihood loss (survival bin)
        alpha=alpha,
        #controls value of ranking loss (discrimination based on feature)
        sigma=sigma,
        duration_index=labtrans.cuts #bin edges
    )
    
    count=0
    subset_X_train_scaled = X_train_scaled[train_idx]
    subset_y_train_discrete = (y_train_discrete[0][train_idx],y_train_discrete[1][train_idx])
    subset_X_test_scaled = X_train_scaled[test_idx]
    subset_y_test_discrete = (y_train_discrete[0][test_idx],y_train_discrete[1][test_idx])

    log = model.fit(
        subset_X_train_scaled,
        subset_y_train_discrete,
        #how many samples per weight update (ie train on chunks of 1024 patients)
        batch_size=1024,
        epochs=3,
        #show (print) training progress
        verbose=True
    )
    surv = model.predict_surv_df(subset_X_test_scaled)
    view_surv = surv
    # target_time = 120
    # closest_idx = np.abs(surv.index - target_time).argmin()
    # surv_at_120 = surv.iloc[closest_idx, :]
    # view_surv_120 = surv_at_120
    # risk_labels = surv_at_120.apply(assign_risk_label)

    integrated_survival = surv.mean(axis=0).values
    risk_labels = [assign_risk_label(p) for p in integrated_survival]

    subset_X_test_extended = subset_X_test_scaled.copy()
    subset_X_test_extended = pd.DataFrame(subset_X_test_scaled,columns=X_test.columns)
    subset_X_test_extended['Risk_Label'] = risk_labels
    print("counts of each risk label are: ")
    print(subset_X_test_extended['Risk_Label'].value_counts())
    view_failed_X = subset_X_test_extended
    view_failed_y = subset_y_test_discrete
    subset_X_test_extended['Survival_Time'] = subset_y_test_discrete[0]
    subset_X_test_extended['Event_Status'] = subset_y_test_discrete[1]

    km_plotter = KM_Plotter(subset_X_test_extended)
    print("checkpoint 1")
    km_plotter.makeKMPlot(plot=True)
    print("checkpoint 2")
    score = km_plotter.getKMScore()
    print("checkpoint 3")
    scores.append(score)
    print("checkpoint 4")
    print(f"Score for fold is: {score}")
    return(mean(scores))

import skopt

cv = RepeatedStratifiedKFold(n_splits=2,n_repeats=1,random_state=1)
best_scores = []
best_configs = [] 
#apply the splits 
for train_idx, test_idx in cv.split(X_train_scaled,y_train_discrete[0]):

    result = skopt.gp_minimize(
        lambda params: evaluate_model(params, train_idx, test_idx),
        search_space,
        n_jobs=3,
        n_calls=50,
        random_state=42
    )
    best_scores.append(result.fun)
    best_configs.append(result.x)

In [ ]:
print(best_scores)
print(best_configs)

In [ ]:
#dropout,lr,alpha,sigma,num_nodes

import skopt.space.space as s

search_space = list()

search_space.append(s.Real(0.001,0.5,name='dropout'))
search_space.append(s.Real(0.001,0.1,name='lr'))
search_space.append(s.Real(0.001,1,name='alpha'))
search_space.append(s.Real(0.001,1,name='sigma'))
search_space.append(s.Categorical(['128,64','256,128,64','256,128,64,32','512,256,128'], name='num_nodes'))

#gives precision score
from sklearn.model_selection import cross_val_score
#for cross validation
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from numpy import mean
from numpy import std
import numpy as np
import sys
import importlib
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

#To view in case something goes wrong
view_failed_X = pd.DataFrame()
view_failed_X_scaled = pd.DataFrame()
view_failed_y_discrete = pd.DataFrame()
view_failed_y = pd.DataFrame()
view_surv = pd.DataFrame()
view_surv_120 = pd.DataFrame()

def evaluate_model(params, train_idx, test_idx):
    num_curves = 5
    global view_failed_X,view_failed_y,view_surv,view_failed_X_scaled,view_failed_y_discrete,view_surv_120
    dropout,lr,alpha,sigma,num_nodes = params
    #creates neural network with fully connected layers
    net = tt.practical.MLPVanilla(
        in_features=number_of_features,
        #hidden layer sizes
        num_nodes=[int(x) for x in num_nodes.split(',')], #output is 101 neurons - one per bin
        out_features=labtrans.out_features, #101 - number of output neurons
        batch_norm=True, #normalise activation between layers - output values of neurons
        dropout=dropout #randomly turn of 30% neurons in each layer to reduce overfitting
    )
    model = DeepHitSingle(
        net,
        #Optimiser that trains neural network by adjusting its weights
        #Uses momentum and RMSprop
        tt.optim.Adam(lr=lr), #lr controls how big the weight updates are
        
        #hyper parameter that controls the value of likelihood loss (survival bin)
        alpha=alpha,
        #controls value of ranking loss (discrimination based on feature)
        sigma=sigma,
        duration_index=labtrans.cuts #bin edges
    )
    
    count=0
    subset_X_train_scaled = X_train_scaled[train_idx]
    subset_y_train_discrete = (y_train_discrete[0][train_idx],y_train_discrete[1][train_idx])
    subset_X_test_scaled = X_train_scaled[test_idx]
    subset_y_test_discrete = (y_train_discrete[0][test_idx],y_train_discrete[1][test_idx])

 
    log = model.fit(
        subset_X_train_scaled,
        subset_y_train_discrete,
        #how many samples per weight update (ie train on chunks of 1024 patients)
        batch_size=1024,
        epochs=3,
        #show (print) training progress
        verbose=True
    )
    surv = model.predict_surv_df(subset_X_train_scaled)
    view_surv = surv
    # target_time = 120
    # closest_idx = np.abs(surv.index - target_time).argmin()
    # surv_at_120 = surv.iloc[closest_idx, :]
    # view_surv_120 = surv_at_120
    # risk_labels = surv_at_120.apply(assign_risk_label)

    integrated_survival = surv.mean(axis=0).values
    min_score, max_score = integrated_survival.min(), integrated_survival.max()
    bins = np.linspace(min_score, max_score, num_curves+1) 
    bins[0] = -np.inf
    bins[-1] = np.inf
    
    #assign the validation samples to corresponding bins, thus assigning labels
    val_scores = model.predict(subset_X_test_scaled)
    print(f"Bins are: {bins}")
    
    try:
        val_risk_groups = pd.cut(val_scores, bins=bins, labels=range(1, num_curves+1), include_lowest=True)
    except Exception as e:
        print(f"Error in binning: {e}")
        print(f"bins were: {bins}")
        return 1000

    subset_X_test_extended = subset_X_test_scaled.copy()
    subset_X_test_extended = pd.DataFrame(subset_X_test_scaled,columns=X_test.columns)
    subset_X_test_extended['Risk_Label'] = val_risk_groups.map(lambda x: f'EL{x}')
    
    print("counts of each risk label are: ")
    print(subset_X_test_extended['Risk_Label'].value_counts())
    view_failed_X = subset_X_test_extended
    view_failed_y = subset_y_test_discrete
    subset_X_test_extended['Survival_Time'] = subset_y_test_discrete[0]
    subset_X_test_extended['Event_Status'] = subset_y_test_discrete[1]

    km_plotter = KM_Plotter(subset_X_test_extended)
    print("checkpoint 1")
    km_plotter.makeKMPlot(plot=True)
    print("checkpoint 2")
    score = km_plotter.getKMScore()
    print("checkpoint 3")
    scores.append(score)
    print("checkpoint 4")
    print(f"Score for fold is: {score}")
    return score 

import skopt

cv = RepeatedStratifiedKFold(n_splits=2,n_repeats=1,random_state=1)
best_scores = []
best_configs = [] 
#apply the splits 
for train_idx, test_idx in cv.split(X_train_scaled,y_train_discrete[0]):

    result = skopt.gp_minimize(
        lambda params: evaluate_model(params, train_idx, test_idx),
        search_space,
        n_jobs=3,
        n_calls=50,
        random_state=42
    )
    best_scores.append(result.fun)
    best_configs.append(result.x)

In [ ]:
# droupout,lr,alpha,sigma,num_nodes = best_configs[0]
# net = tt.practical.MLPVanilla(
#         in_features=number_of_features,
#         #hidden layer sizes
#         num_nodes=[int(x) for x in num_nodes.split(',')], #output is 101 neurons - one per bin
#         out_features=labtrans.out_features, #101 - number of output neurons
#         batch_norm=True, #normalise activation between layers - output values of neurons
#         dropout=droupout #randomly turn of 30% neurons in each layer to reduce overfitting
# )
# model = DeepHitSingle(
#     net,
#     #Optimiser that trains neural network by adjusting its weights
#     #Uses momentum and RMSprop
#     tt.optim.Adam(lr=lr), #lr controls how big the weight updates are
    
#     #hyper parameter that controls the value of likelihood loss (survival bin)
#     alpha=alpha,
#     #controls value of ranking loss (discrimination based on feature)
#     sigma=sigma,
#     duration_index=labtrans.cuts #bin edges
# )
# log = model.fit(
#     X_train_scaled,
#     y_train_discrete,
#     #how many samples per weight update (ie train on chunks of 1024 patients)
#     batch_size=1024,
#     epochs=3,
#     #show (print) training progress
#     verbose=True
# )

# surv = model.predict_surv_df(X_test_scaled)

# X_test_extended = X_test_scaled.copy()

# integrated_survival = surv.mean(axis=0).values
# risk_labels = [assign_risk_label(p) for p in integrated_survival]

# X_test_extended = pd.DataFrame(X_test_scaled,columns=X_test.columns)
# X_test_extended['Risk_Label'] = risk_labels

# X_test_extended['Survival_Time'] = durations_test
# X_test_extended['Event_Status'] = events_test

# km_plotter = KM_Plotter(X_test_extended)

# km_plotter.makeKMPlot(plot=True)
# print(f"KM score is: {km_plotter.getKMScore()}")

In [ ]:
droupout,lr,alpha,sigma,num_nodes = best_configs[1]
num_curves = 6
net = tt.practical.MLPVanilla(
        in_features=number_of_features,
        #hidden layer sizes
        num_nodes=[int(x) for x in num_nodes.split(',')], #output is 101 neurons - one per bin
        out_features=labtrans.out_features, #101 - number of output neurons
        batch_norm=True, #normalise activation between layers - output values of neurons
        dropout=droupout #randomly turn of 30% neurons in each layer to reduce overfitting
)
model = DeepHitSingle(
    net,
    #Optimiser that trains neural network by adjusting its weights
    #Uses momentum and RMSprop
    tt.optim.Adam(lr=lr), #lr controls how big the weight updates are
    
    #hyper parameter that controls the value of likelihood loss (survival bin)
    alpha=alpha,
    #controls value of ranking loss (discrimination based on feature)
    sigma=sigma,
    duration_index=labtrans.cuts #bin edges
)
log = model.fit(
    X_train_scaled,
    y_train_discrete,
    #how many samples per weight update (ie train on chunks of 1024 patients)
    batch_size=1024,
    epochs=3,
    #show (print) training progress
    verbose=True
)

surv = model.predict_surv_df(X_train_scaled)

X_test_extended = X_test_scaled.copy()

integrated_survival = surv.mean(axis=0).values

min_score, max_score = integrated_survival.min(), integrated_survival  .max()
bins = np.linspace(min_score, max_score, num_curves+1) #4 edges = 3 bins
bins[0] = -np.inf
bins[-1] = np.inf

val_scores = model.predict_surv_df(X_test_scaled).mean(axis=0).values


val_risk_groups = pd.cut(val_scores, bins=bins, labels=range(num_curves,0,-1), include_lowest=True)


X_test_extended = pd.DataFrame(X_test_scaled,columns=X_test.columns)
X_test_extended['Risk_Label'] = val_risk_groups.map(lambda x: f'EL{x}')

X_test_extended['Survival_Time'] = durations_test
X_test_extended['Event_Status'] = events_test

km_plotter = KM_Plotter(X_test_extended,num_curves=num_curves)

km_plotter.makeKMPlot(plot=True)
print(f"KM score is: {km_plotter.getKMScore()}")

In [ ]:
#Test new risk label assignment strategy based np.bin stratification
print(best_configs)
print(best_scores)

In [ ]:
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

def km_scorer(estimator, X, y):
    X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    surv = estimator.predict_surv_df(X_train_sub)
    integrated_survival = surv.mean(axis=0).values

    min_score, max_score = integrated_survival.min(), integrated_survival.max()
    bins = np.linspace(min_score, max_score, num_curves + 1)
    bins[0] = -np.inf
    bins[-1] = np.inf

    integrated_survival_test = estimator.predict_surv_df(X_test_sub).mean(axis=0).values
    risk_groups = pd.cut(
        integrated_survival_test,
        bins=bins,
        labels=range(num_curves, 0, -1),
        include_lowest=True
    )

    
    X_test_extended = pd.DataFrame({
    'Risk_Label': risk_groups.map(lambda x: f'EL{x}'),
    'Survival_Time': y_test_sub['time'],
    'Event_Status': y_test_sub['event'].astype(np.float32)})

    km_plotter = KM_Plotter(X_test_extended, num_curves=num_curves)
    km_plotter.makeKMPlot(plot=False)

    return 1 - km_plotter.getKMScore()


droupout,lr,alpha,sigma,num_nodes = best_configs[1]
num_curves = 6
net = tt.practical.MLPVanilla(
        in_features=number_of_features,
        #hidden layer sizes
        num_nodes=[int(x) for x in num_nodes.split(',')], #output is 101 neurons - one per bin
        out_features=labtrans.out_features, #101 - number of output neurons
        batch_norm=True, #normalise activation between layers - output values of neurons
        dropout=droupout #randomly turn of 30% neurons in each layer to reduce overfitting
)
model = DeepHitSingle(
    net,
    #Optimiser that trains neural network by adjusting its weights
    #Uses momentum and RMSprop
    tt.optim.Adam(lr=lr), #lr controls how big the weight updates are
    
    #hyper parameter that controls the value of likelihood loss (survival bin)
    alpha=alpha,
    #controls value of ranking loss (discrimination based on feature)
    sigma=sigma,
    duration_index=labtrans.cuts #bin edges
)
log = model.fit(
    X_train_scaled,
    y_train_discrete,
    #how many samples per weight update (ie train on chunks of 1024 patients)
    batch_size=1024,
    epochs=3,
    #show (print) training progress
    verbose=True
)


# Calculate permutation importance
result = permutation_importance(
    model,                    # trained DeepHit model
    X_test_scaled,           # Test features (scaled)
    y_test,                  # Test target (structured array)
    n_repeats=10,            # Number of times to permute each feature
    random_state=42,
    scoring=km_scorer,  # Use KM score as metric
    n_jobs=-1                # Use all CPU cores
)

print("\n[SUCCESS] Permutation importance calculated!")

# Create DataFrame with results
perm_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values('importance_mean', ascending=False)

# Show top 15
print("\n" + "="*80)
print("TOP 15 MOST IMPORTANT FEATURES (Permutation Importance)")
print("="*80)
print(perm_importance.head(15).to_string(index=False))


# Statistical significance check
# Features with importance > 2*std are statistically significant
perm_importance['significant'] = perm_importance['importance_mean'] > 2 * perm_importance['importance_std']

print(f"\nStatistically significant features: {perm_importance['significant'].sum()}")

# Visualize top 20
import matplotlib.pyplot as plt

top_20 = perm_importance.head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(15), top_20['importance_mean'], xerr=top_20['importance_std'])
plt.yticks(range(15), top_20['feature'])
plt.xlabel('Permutation Importance (Change in KM score)')
plt.title('Top 15 Feature Importances - Permutation Method')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('deephit_permutation_importance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plotNPHistogram(array,bins,title):
    # Get histogram data
    counts, bin_edges = np.histogram(array,bins)

    # Plot as bar chart
    plt.figure(figsize=(10, 6))
    plt.bar(
        range(len(counts)), 
        counts, 
        tick_label=['0-20', '20-40', '40-60', '60-80', '80-100', 
                    '100-120', '120-140', '140-160', '160-180', '180-200']
    )
    plt.xlabel('Survival Time (months)', fontsize=12)
    plt.ylabel('Number of Patients', fontsize=12)
    plt.title(title, 
          fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
def getELStats(EL_Label):
    EL_df = X_test_extended[X_test_extended['Risk_Label'] == EL_Label]
    EL_alive = EL_df[EL_df['Event_Status'] == 0]
    EL_dead = EL_df[EL_df['Event_Status'] == 1]

    print(f"Number of event occurences: {(EL_df['Event_Status'] == 0).sum()}")
    print(f"Number of censored occurences: {(EL_df['Event_Status'] == 1).sum()}")
    plotNPHistogram(EL_alive['Survival_Time'],[0,20,40,60,80,100,120,140,160,180,200],"Survival Months for Censored Patients")
    plotNPHistogram(EL_dead['Survival_Time'],[0,20,40,60,80,100,120,140,160,180,200],"Survival Months for Event Patients")
  
getELStats("EL6")

In [ ]:
print(y_train["time"])
plt.figure(figsize=(12, 6))
bins = range(0, 130, 10)  # 0, 10, 20, ..., 120
plt.hist(y_train["time"], bins=bins, edgecolor='black', alpha=0.7, color='steelblue')

plt.xlabel('Duration (months)', fontsize=12)
plt.ylabel('Number of Patients', fontsize=12)
plt.title('Distribution of Survival Duration in Train Set', fontsize=14, fontweight='bold')
plt.xticks(bins)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('Report_Resources/y_test_duration_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

#print frequency for each bin
# Get histogram counts and bin edges
counts, bin_edges = np.histogram(y_train["time"], bins=range(0, 130, 10))

# Print frequency for each bin
for i in range(len(counts)):
    print(f"{bin_edges[i]}-{bin_edges[i+1]} months: {counts[i]} patients")

print(f"Total # of Patients: {len(y_train)}")

In [ ]:
#Alternate Classification System, using exact 119 month time point

#Get survival predictions for test set
surv = model.predict_surv_df(X_test_scaled)

#each column is a patient's predicted survival curve over time
#each row is a time 
#Transpose makes more sense
#cell in table - probability patient survives at a specific time
print(f"surv is {surv}")
#probability patient 1 survives x months for an array of xs
#print(surv[1])

#Get probability patient survives 10 years
time_point = 10  #months

#Gets closest time point in the models discrete time grid, 
#get index of smallest distance - closest to expected survival
#gets the bin (row label of .predict_surv_df) closest to 120 months
closest_idx = np.abs(model.duration_index - time_point).argmin() 

#extract probability patient survives 10 years
survival_probs = surv.iloc[closest_idx].values

print(np.histogram(survival_probs,bins=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]))
#evaluate how true it is that no patient survives ten years
print(np.histogram(durations_test,bins=[0,20,40,60,80,100,120,140,160,180,200]))
# print(f"row is: {closest_idx}") #43
# print(f"bin (months) is: {model.duration_index[43]}") #119

#Apply labels
risk_labels = [assign_risk_label(p) for p in survival_probs]

#Add survival months
print(f"risk_labels shape: {len(risk_labels)}")


# Create extended dataframe
X_test_extended = pd.DataFrame(X_test_scaled,columns=X_test.columns)
print(f"X_test_extended shape: {X_test_extended.shape}")
      
#X_test_extended['Risk_Label'] = risk_labels
X_test_extended['Risk_Label'] = risk_labels
X_test_extended['Survival_Time'] = durations_test
X_test_extended['Event_Status'] = events_test

#check distribution
print(X_test_extended['Risk_Label'].value_counts().sort_index())
makeKMPlot(X_test_extended)

In [ ]:
#DeepHit Curves
# Instead of KM plot, plot model predictions directly
plt.figure(figsize=(14, 8))

for label in ['EL1', 'EL2', 'EL3', 'EL4','EL5', 'EL6', 'EL7', 'EL8', 'EL9','EL10']:
    # Get patients in this risk group
    mask = X_test_extended['Risk_Label'] == label
    patient_indices = np.where(mask)[0]
    
    # Average their predicted survival curves
    avg_surv = surv.iloc[:, patient_indices].mean(axis=1)
    
    # Plot
    plt.plot(model.duration_index, avg_surv, label=f'{label} (n={mask.sum()})', linewidth=2.5)

plt.xlabel('Time (months)')
plt.ylabel('Predicted Survival Probability')
plt.title('Model-Predicted Survival Curves by Risk Group')
plt.legend()
plt.xlim(0, 120)
plt.show()

In [ ]:
%pip install lifelines

#expect this line to be like this since patients will die
#log level KM Plot
#Input and output needs modification 

#Transform input to get differnt results
#Want to explain which features are most important and why
#Take first 15 years to be training data and last 5 years to be testing data
#External factors like pandemic has other effects
#Cannot expect to have result we want 
#Try regression as opposed to classification
#Try XGBoost -FIRST - 
#algorithm/model is not important, fluent structure of paper is more important
#choose model based on how well it works for data 

#Read papers around algorithms for survival analysis
#Look into converting data into an image and try image classification (FMRI to brain region id prediction)
#Use clustering to fill in missing data to fill in and normalise before passing to model
#Normalise data first - remove outliers (FIRST)
#Use synthetic data - create diffusion model to make synthetic data

#Investigate neural networks
#Look into clustering - KNN with k = 10 for 10 stages, can get important score of features
#Need to just investigate reason as why KM plot looks so steep, splits are more finegrained, as a consequence not enough data for each label
#outliers may have bigger impact because of little data in each label
#bugs in my code - and redesign input and output 

#A good deep learning model requires good data and resources - main cause of bad results is because of lack of data
#Explain importance of factors for deep learning model 
#XGBoost good because it's easier and I can focus on data investigation and improvement
#Redesign (transformation) of input and output is something I need to always do. 

#Start by investigating problems by myself, if I am still confused send Minghao email




In [ ]:
print(f"Time point: {time_point} months")
print(f"Index used: {closest_idx}")
print(f"Actual time: {model.duration_index[closest_idx]:.1f} months")
print(X_test_extended['Risk_Label'].value_counts().sort_index())
print(f"Event rate: {events_test.mean():.3f}")
print(f"Events: {events_test.sum()} / {len(events_test)}")

In [ ]:
#Implement Bootstrapping
 
B = 10
n_train = len(X_train)

for b in range(B):
    print(f'\n--- Bootstrap iteration {b + 1}/{B} ---')

    rng = np.random.RandomState(seed=b)
    boot_idx = rng.randint(0, len(X_train), size=len(X_train))

    X_boot = X_train_scaled[boot_idx]
    dur_boot = durations_train[boot_idx]
    evt_boot = events_train[boot_idx]

    print(f'  Unique samples: {len(np.unique(boot_idx))}/{n_train} '
            f'({len(np.unique(boot_idx)) / n_train:.1%})')

    labtrans = LabTransDiscreteTime(num_durations)
    y_boot_discrete = labtrans.fit_transform(dur_boot, evt_boot)



    model.fit(
        X_boot,
        y_boot_discrete,
        batch_size= fit_config['batch_size'],
        epochs= fit_config['epochs'],
        verbose= fit_config['verbose'],
    )

           

    def predict_surv_df(self, X_test_scaled):
        """
        Average survival predictions across all bootstrap models.

        Returns a DataFrame with the time grid of the first model as index.
        """
        surv_list = [m.predict_surv_df(X_test_scaled) for m in self.models]

        # Align all to the first model's time index
        ref_index = surv_list[0].index
        aligned = []
        for s in surv_list:
            s_reindexed = s.copy()
            s_reindexed.index = ref_index
            aligned.append(s_reindexed)

        return sum(aligned) / len(aligned)